# 101 - Zernikes

prysm has a vast polynomials library -- it is nearly the size of ther rest of
the core library combined -- and supports just about every polynomial you can
imagine.  All of them have analytic derivatives and optimized implementations
based on recurrence relations or other more advanced mathematical techniques.

However, in optics roughly 99% of the time you will work with Zernike
polynomials.  prysm works with them in perhaps a slightly different interface
than you may be used to.  In this tutorial, we'll show how the interface works
and why it is in the style that it is.

----

The Zernikes are dual index basis functions $Z_n^m$ basis.  In almost all cases,
you do not work with the dual index and instead use an indexing scheme - fringe,
Noll, Born & Wolf, ANSI j - etc.

In many other packages, you have something
like `def zernike_compose(coefs) -> ndarray` where coefs must be a dense, Noll
ordered set of coefficients.  Or `def zernike_fringe(coefs) -> ndarray`, and so
on.  in prysm, this is broken into three steps:

1.  `def fringe_to_nm()`
2.  `def zernike_nm()`
3. `def sum_of_2d_modes()`

you compute the `(n,m)` indices, then the basis functions, then computed weighted
sums of those basis functions.  This has several benefits:

- adding a new indexing scheme does not require duplicating the other two steps
- computing the basis functions, which is significantly more computationally
  intensive, happens just once
- weighted sums of zernikes and weighted sums of other polynomials "fan in" to
  the same code

We'll do our usual imports and show how to work with zernikes:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from prysm.coordinates import make_xy_grid, cart_to_polar
from prysm.geometry import circle
# you would usually do polynomials.xxx, but we will walk through
# much more of the Zernike API than you would normally, so we'll do
# zernike.xxx instead
from prysm.polynomials import zernike, sum_of_2d_modes, lstsq

In [ ]:
# set up a simple grid and mask to NaN things outside
# the unit circle

x, y = make_xy_grid(256, diameter=2)
r, t = cart_to_polar(x, y)
out = ~circle(1, r)

def show(arr, ax=None, blank=True, **kw):
    a = np.asarray(arr, dtype=float).copy()
    if blank:
        a[out] = np.nan
    ax = ax or plt.gca()
    return ax.imshow(a, cmap=kw.pop('cmap', 'RdBu'), **kw)

## Indexing schemes

Three indexing schemes are supported (Born & Wolf may come in a future update).  The backwards mapping (n,m) -> j is available for fringe and ANSI orderings, but not for Noll at this time.

In [ ]:
print(f"{'index':>6} {'Noll (n,m)':>12} {'Fringe (n,m)':>14} {'ANSI (n,m)':>12}")
for j in range(1, 17):
    print(f'{j:>6} {str(zernike.noll_to_nm(j)):>12} {str(zernike.fringe_to_nm(j)):>14} {str(zernike.ansi_j_to_nm(j)):>12}')

## Generating modes

There are also several ways to generate Zernike modes, or their spatial
derivatives.  The most common thing is to run one of the indexing schemes
in a loop to make a list of `[(n,m)]`, then use one of the _sequence_ functions
to generate the cube of basis modes.  This is done because the recurrence
relations used in the polynomial module in general use `P_(n-2)` and `P_(n-1)`
to calculate `P_n`, so the sequence avoids duplicate work by generating all
of the modes in a single pass.  This is a little less true for the Zernikes
because of the complex nature of their $R_n^m \cdot \cos(m*\theta)$ form, but
some speedup is still achieved.

We'll show the two main functions: `zernike_nm` and `zernike_nm_seq`.  A
reminder that the masking outside the unit circle is done by the `show`
function, and prysm will compute and return the correct but non-orthogonal
value of the polynomials outside of the unit circle, instead of returning
0, NaN, or another sentinel value.

In [ ]:
z_coma = zernike.zernike_nm(3, 1, r, t, norm=True)  # (n=3, m=1) = coma
z_sph = zernike.zernike_nm(4, 0, r, t, norm=True)   # (n=4, m=0) = spherical
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
show(z_coma, axs[0]); axs[0].set(title='Zernike coma (3, 1)', xticks=[], yticks=[])
show(z_sph, axs[1]); axs[1].set(title='Zernike spherical (4, 0)', xticks=[], yticks=[])

## basis sets and sums

For repeated sums (for example, simulating many wavefront perturbations to a
system), precompute a basis:

In [ ]:
nms = [zernike.noll_to_nm(j) for j in range(1,37)]
basis = zernike.zernike_nm_seq(nms, r, t, norm=True)

# random numbers, decreasing in magnitude with index
coef = np.random.rand(36) * 1/(1+np.arange(36))
coef[:3] = 0 # zero piston tip tilt

phs = sum_of_2d_modes(basis, coef)

show(phs);
plt.title('OPD map from Zernike sum');

fig, axs = plt.subplots(5, 6, figsize=(12, 12))
for ax, mode, (n, m) in zip(axs.ravel(), basis, nms):
    show(mode, ax, blank=True)
    ax.set(title=f'({n},{m})', xticks=[], yticks=[])
fig.suptitle('the first 36 Noll Zernikes')
fig.tight_layout()

## Fitting

Of course, the next useful thing to do with a basis is fit coefficients
to data.  The polynomials module also has a lstsq function, "least squares"
which **does not** assume orthogonality and uses a singular value decomposition
approach to regularize the fit.  Like `sum_of_2d_modes`, lstsq is used for all
polynomial bases in exactly the same way.  It is NaN-aware, you can use
NaN to mask pixels that should not contribute to the fit.

In [ ]:
from prysm.polynomials import lstsq, sum_of_2d_modes

fit = lstsq(basis, phs)
rmse = np.linalg.norm(fit - coef)
print('RMS error of fit coefficients:', rmse)

# mask data
phs[::2,::2]=np.nan
fit = lstsq(basis, phs)
rmse = np.linalg.norm(fit - coef)
print('after masking every 2nd pixel')
print('RMS error of fit coefficients:', rmse)

## Wrap-up

This tutorial walked through the various Zernike indexing schemes, building
basis sets, and least squares fitting.  The next tutorials will cover
orthogonalizing polynomials over annular and arbitrary domains (102), all
of the other polynomial bases (103), Q type polynomials and Clenshaw's
method for computing weighted sums in a single pass (for raytracing where
small changes in position mean a precomputed basis set is useless) (104),
and analytical derivatives in 201.